# SpatialMemory — Before/After Upload Workspace

Run the two cells below to create a temporary password-protected upload link.
The existing neural pipeline runs on **CPU**, including when a GPU runtime is selected.
Free Colab memory/session availability is not guaranteed. Start with short recordings.

**Research use:** VGGT weights carry a noncommercial research restriction. This notebook does not grant commercial-use rights. Read [model licences](https://github.com/VamshiKrishnaBandari07/Spatial-AI-system-for-Humanoid-Robotics/blob/main/MODEL_REGISTRY.md) before accepting.

Recordings are uploaded to your Colab runtime and browser traffic is relayed by Gradio. They are not sent to remote model inference APIs. Download results before ending the session. UNKNOWN is an abstention, not a confirmed change or an accuracy estimate.


In [ ]:
# Step 1 — Accept applicable research terms and install (first run takes time).
ACCEPT_RESEARCH_TERMS = False #@param {type:"boolean"}
if not ACCEPT_RESEARCH_TERMS:
    raise RuntimeError("Read the checkpoint licences above; accept only if appropriate for your use.")

import os, shutil, subprocess, sys
from pathlib import Path
REPO = Path("/content/Spatial-AI-system-for-Humanoid-Robotics")
VENV = Path("/content/spatialmemory-colab-venv")
def checked(*args, **kwargs):
    return subprocess.run(list(args), check=True, **kwargs)
if not REPO.exists():
    checked("git", "clone", "--depth", "1", "https://github.com/VamshiKrishnaBandari07/Spatial-AI-system-for-Humanoid-Robotics.git", str(REPO))
print("Repository revision:")
checked("git", "-C", str(REPO), "rev-parse", "HEAD")
checked("free", "-h")
checked("df", "-h", "/content")
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    checked("apt-get", "update", "-qq")
    checked("apt-get", "install", "-y", "ffmpeg")
if not VENV.exists():
    checked(sys.executable, "-m", "venv", str(VENV))
PYTHON = str(VENV / "bin/python")
checked(PYTHON, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0", "--index-url", "https://download.pytorch.org/whl/cpu")
checked(PYTHON, "-m", "pip", "install", str(REPO) + "[neural]", "gradio==5.49.1", "huggingface_hub==0.36.2")

# Core loaders retain their existing /workspace paths and pinned provenance checks.
Path("/workspace/spatialmemory-thirdparty").mkdir(parents=True, exist_ok=True)
for name, url, revision in (
    ("vggt", "https://github.com/facebookresearch/vggt.git", "a288dd0f14786c93483e45524328726ab7b1b4ce"),
    ("sam2", "https://github.com/facebookresearch/sam2.git", "2b90b9f5ceec907a1c18123530e92e794ad901a4"),
):
    source = Path("/workspace/spatialmemory-thirdparty") / name
    if not source.exists():
        source.mkdir()
        checked("git", "-C", str(source), "init")
        checked("git", "-C", str(source), "remote", "add", "origin", url)
        checked("git", "-C", str(source), "fetch", "--depth", "1", "origin", revision)
        checked("git", "-C", str(source), "checkout", "--detach", "FETCH_HEAD")
    actual = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()
    if actual != revision:
        raise RuntimeError(f"{name} source mismatch; existing files were preserved.")
    checked("git", "-C", str(source), "diff", "--quiet")
ENV = dict(os.environ, HF_HOME="/workspace/spatialmemory-models/hf-home", HF_XET_CACHE="/workspace/spatialmemory-models/xet", OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")
checked(PYTHON, "scripts/download_models.py", cwd=str(REPO), env=ENV)
checked(PYTHON, "-m", "pip", "check")
print("Installation finished. Run Step 2 for your upload link.")


In [ ]:
# Step 2 — Launch; the password is entered privately and never saved in this notebook.
import getpass, os, re, subprocess, time
from pathlib import Path

# Reconstruct launch settings even after notebook variables are cleared.
REPO = Path("/content/Spatial-AI-system-for-Humanoid-Robotics")
PYTHON = "/content/spatialmemory-colab-venv/bin/python"
if not Path(PYTHON).is_file() or not (REPO / "scripts/launch_colab.py").is_file():
    raise RuntimeError("Setup is missing. Run Step 1 successfully before launching the UI.")
check = subprocess.run(
    [PYTHON, "-c", "import gradio, torch, transformers, spatialmemory.colab_ui"],
    capture_output=True, text=True,
)
if check.returncode:
    raise RuntimeError("Setup is incomplete. Rerun Step 1 and resolve its error before launching.\n" + check.stderr[-2000:])
if "UI_PROCESS" in globals() and UI_PROCESS.poll() is None:
    raise RuntimeError("Your UI is already running. Use its existing link or stop it below.")
password = getpass.getpass("Choose a UI password (at least 12 characters): ")
if len(password) < 12:
    raise ValueError("Use at least 12 characters.")
launch_env = dict(
    os.environ,
    HF_HOME="/workspace/spatialmemory-models/hf-home",
    HF_XET_CACHE="/workspace/spatialmemory-models/xet",
    OMP_NUM_THREADS="4", MKL_NUM_THREADS="4",
    SPATIALMEMORY_UI_PASSWORD=password,
    SPATIALMEMORY_RUNS="/content/spatialmemory-runs", PYTHONUNBUFFERED="1",
)
LOG = Path("/content/spatialmemory-ui.log")
with LOG.open("w") as log:
    UI_PROCESS = subprocess.Popen([PYTHON, "scripts/launch_colab.py"], cwd=str(REPO), env=launch_env, stdout=log, stderr=subprocess.STDOUT)
del password, launch_env
for attempt in range(120):
    text = LOG.read_text()
    match = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", text)
    if match:
        print("Open your upload UI:", match.group(0))
        print("Username: reviewer | Password: the one you just entered")
        print("Upload BEFORE and AFTER separately. Save results.zip before disconnecting.")
        break
    if UI_PROCESS.poll() is not None:
        raise RuntimeError("UI startup failed. Log:\n" + text)
    time.sleep(1)
else:
    print("No public link received yet. Inspect /content/spatialmemory-ui.log; relay access may be unavailable.")


## Review results

Keep **Neural + objects** selected for VGGT, Grounding DINO, SAM 2 and DINOv2 inference. Geometry diagnostic is explicitly non-neural. Processing may take several minutes. Registration failure blocks change claims; no identity fallback is forced. Missing validated movement error budgets preserve UNKNOWN.

Download and extract **results.zip**, then open **visualization/index.html** for the interactive 3D scene and evidence. Retain JSON/PLY/diagnostics for review. Ground-truth changes must come from independently reviewing original recordings, never from copying predictions.


In [ ]:
# Optional — Stop only the UI process started by this notebook.
# Wait for an active comparison to finish before running this cell.
if "UI_PROCESS" in globals() and UI_PROCESS.poll() is None:
    UI_PROCESS.terminate()
    UI_PROCESS.wait(timeout=30)
    print("UI stopped. Downloaded results remain on your computer; runtime files remain until Colab clears them.")
